<a id="inizio"></a>
# Approfondimenti 2 · Soluzioni

⏱ ~60 min &nbsp;·&nbsp; Dati: `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/impianti_fv.csv`

**In questo notebook impariamo a**

- misurare durate con `Timedelta`, gestire il fuso orario e costruire lag e medie mobili con `shift` e `rolling`
- navigare una serie con range slider e range selector ed esportare il grafico in HTML
- riconoscere type hint, `@dataclass`, decoratori, generatori e `**kwargs`

Le sezioni sulle date usano `df_15min`, la serie del notebook 08: potenza e temperatura inventate,
un valore ogni 15 minuti per tre giorni, con i buchi riempiti dall'interpolazione lineare. La ricostruiamo qui.

In [ ]:
import numpy as np
np.random.seed(42)
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
intervallo = pd.date_range("2025-03-01 00:00", periods=3 * 24 * 4, freq="15min")
ore = intervallo.hour + intervallo.minute / 60
power_kw = 220 + 60 * np.sin(2 * np.pi * (ore / 24)) + np.random.normal(0, 8, size=len(intervallo))
temp_c = 12 + 5 * np.sin(2 * np.pi * ((ore - 6) / 24)) + np.random.normal(0, 0.7, size=len(intervallo))

misure = pd.DataFrame({"timestamp": intervallo, "power_kw": power_kw.round(1), "temp_c": temp_c.round(1)})
misure = misure.drop(np.random.choice(misure.index, size=18, replace=False))  # i buchi

df_15min = misure.set_index("timestamp").asfreq("15min").interpolate(method="linear")
df_15min.head()

<a id="sez-1"></a>
## 1. Differenze tra date

Le differenze tra timestamp producono `Timedelta` (durate).
Servono sia per capire la regolarità della serie sia per misurare intervalli (es. "quante ore copre il dataset?").

In [ ]:
# differenze tra timestamp consecutivi
delta_consecutivi = df_15min.index.to_series().diff()

delta_consecutivi.head(10)

In [ ]:
# intervallo totale coperto
delta_totale = df_15min.index.max() - df_15min.index.min()
delta_totale

In [ ]:
# durata in ore (float)
delta_totale.total_seconds() / 3600

<a id="sez-2"></a>
## 2. Fuso orario

Un timestamp può essere **naive**, senza fuso orario (solo data e ora), oppure **timezone-aware**,
con fuso orario (es. `Europe/Rome`). Spesso si ricevono orari locali naive da interpretare correttamente.
L'ora legale è la trappola tipica: alcune ore non esistono (primavera) o si ripetono (autunno).

Documentazione: [fusi orari](https://pandas.pydata.org/pandas-docs/stable/user_guide/timeseries.html#time-zone-handling)

<div style="background:rgba(217,83,79,0.12); border-left:4px solid #d9534f; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**⚠️ Attenzione**

Localizzare un indice che attraversa un cambio ora legale può generare timestamp ambigui o inesistenti:
in quei casi servono i parametri `ambiguous=` e `nonexistent=`.
Se i dati arrivano già in UTC, spesso è meglio mantenerli in UTC e convertire solo in visualizzazione.

</div>

In [ ]:
df_fuso = df_15min.copy()

# interpretiamo i timestamp come orari locali italiani
df_fuso.index = df_fuso.index.tz_localize("Europe/Rome")

# conversione a UTC (utile per sistemi e confronti)
df_utc = df_fuso.tz_convert("UTC")

df_fuso.index[:3], df_utc.index[:3]

In [ ]:
df_fuso.loc["2025-03-01 06:00":"2025-03-01 12:00"].head()

<a id="sez-3"></a>
## 3. Shift e rolling

Due operazioni semplici e molto usate nella gestione di time series:

- `shift(k)`: sposta i valori di `k` step nel tempo (crea "lag").
- `rolling(w)`: calcola statistiche su una finestra mobile di ampiezza `w`.

Entrambe possono introdurre `NaN` in testa (perché mancano i valori "precedenti").
Documentazione: [`rolling`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.rolling.html)

In [ ]:
df_scorrimento = df_15min.copy()

# lag di 1 ora: con frequenza 15min corrisponde a 4 step
df_scorrimento["power_lag_4"] = df_scorrimento["power_kw"].shift(4)

# media mobile su 2 ore: 8 step
df_scorrimento["power_ma_8"] = df_scorrimento["power_kw"].rolling(8).mean()

df_scorrimento[["power_kw", "power_lag_4", "power_ma_8"]].head(12)

In [ ]:
# shift di 1 ora (4 step) contro power_kw
fig = px.line(df_scorrimento, x=df_scorrimento.index, y=["power_kw", "power_lag_4"], title="Shift di 1 ora (4 step)")
fig.show()

In [ ]:
# media mobile su una finestra di 2 ore (8 step) contro power_kw
fig = px.line(df_scorrimento, x=df_scorrimento.index, y=["power_kw", "power_ma_8"], title="Media mobile su finestra di 2 ore (8 step)")
fig.show()

<a id="sez-4"></a>
## 4. Range slider e range selector

Qui si passa a `graph_objects` per usare il **range slider** e il **range selector** sull'asse x,
utili per navigare serie storiche lunghe. Il codice segue lo schema degli
[esempi ufficiali](https://plotly.com/python/range-slider/): si costruiscono le tracce, poi si
aggiunge il controllo nel `layout`.

Usiamo il dataset `px.data.stocks()` del notebook 09: una colonna tempo (`date`) e una colonna per
azienda, con i valori indicizzati (partono da 1).

In [ ]:
df = px.data.stocks().copy()
df["date"] = pd.to_datetime(df["date"])
df.head()

In [ ]:
aziende = ["AAPL", "AMZN", "MSFT"]

fig = go.Figure()
for c in aziende:
    fig.add_trace(go.Scatter(x=list(df["date"]), y=list(df[c]), name=c))

Il range selector offre pulsanti predefiniti (1m, 6m, 1y, all) per saltare a intervalli comuni. Lo
slider in basso permette di selezionare un intervallo qualsiasi.

In [ ]:
bottoni = [
    dict(count=1, label="1m", step="month", stepmode="backward"),
    dict(count=6, label="6m", step="month", stepmode="backward"),
    dict(count=1, label="1y", step="year", stepmode="backward"),
    dict(step="all"),
]

In [ ]:
# range slider + range selector (schema ufficiale)
fig.update_layout(
    title_text="Stocks (indicizzato), range slider",
    height=400,
    width=700,
    xaxis=dict(
        rangeselector=dict(buttons=bottoni, bgcolor="lightgray", font=dict(size=10)),
        rangeslider=dict(visible=True),
        type="date",
    ),
    margin=dict(t=80, b=80),
)
fig.show()

<a id="sez-5"></a>
## 5. Esportare in HTML

Per condividere il grafico interattivo senza Jupyter, l'export in HTML è la strada più semplice
([documentazione](https://plotly.com/python/interactive-html-export/)).

In [ ]:
# salva l'ultima figura (fig) in HTML, nella cartella del notebook: si apre con un browser
# con include_plotlyjs="cdn" la libreria Plotly si scarica all'apertura: file leggero, ma serve la rete
fig.write_html("plot_time_series.html", include_plotlyjs="cdn")

<a id="sez-6"></a>
## 6. Leggere il codice di un agente: type hint, dataclass, decoratori, generatori, kwargs

Cinque costrutti che compaiono spesso negli script scritti da un agente, come quello letto nel
notebook 10, ognuno con un esempio piccolo.

<a id="sez-6-1"></a>
### 6.1 Type hint

Le forme più frequenti sono `list[str]`, `dict[str, float]`, `X | None` e `-> None`; Python non le
controlla quando esegue.

In [ ]:
def prezzo_medio(prezzi: dict[str, float], escludi: list[str] | None = None) -> float:
    """Prezzo medio dei prodotti, senza quelli in escludi."""
    escludi = escludi or []
    validi = [p for nome, p in prezzi.items() if nome not in escludi]
    return sum(validi) / len(validi)


print(prezzo_medio({"pane": 2.5, "latte": 1.3, "caffè": 4.2}, escludi=["caffè"]))  # Output: 1.9
print(prezzo_medio({"pane": 2, "latte": 1}))  # int al posto di float, nessun errore. Output: 1.5

<a id="sez-6-2"></a>
### 6.2 `@dataclass`

Su una classe fatta di campi, `@dataclass` scrive da solo il costruttore, la stampa e il confronto con `==`.

In [ ]:
from dataclasses import dataclass


@dataclass
class Libro:
    titolo: str
    autore: str
    pagine: int = 0


libro = Libro("Il nome della rosa", "Umberto Eco", pagine=503)
print(libro)
print(libro == Libro("Il nome della rosa", "Umberto Eco", 503))  # Output: True

<a id="sez-6-3"></a>
### 6.3 Decoratori

Una riga `@nome` sopra un `def` avvolge la funzione in un'altra: `@cache` ricorda i risultati già calcolati.

In [ ]:
from functools import cache


@cache  # senza, fibonacci(80) richiederebbe miliardi di chiamate
def fibonacci(n: int) -> int:
    return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)


fibonacci(80)  # altri decoratori frequenti: @property, @staticmethod, @pytest.fixture

<a id="sez-6-4"></a>
### 6.4 Generatori

Una funzione con `yield` al posto di `return` consegna un valore alla volta, quando un `for` o `list()` lo chiede.

In [ ]:
def a_blocchi(elementi: list, n: int):
    for i in range(0, len(elementi), n):
        yield elementi[i:i + n]


spesa = ["pane", "latte", "uova", "mele", "pasta"]
list(a_blocchi(spesa, 2))

<a id="sez-6-5"></a>
### 6.5 `**kwargs`

`**kwargs` raccoglie in un dizionario i parametri passati per nome che la funzione non elenca
(`*args` in una tupla quelli per posizione); negli script degli agenti li passa così come sono a
un'altra funzione.

In [ ]:
def ordine(piatto: str, **opzioni) -> str:
    return f"{piatto}: {opzioni}"


def leggi_csv(path: str, **opzioni) -> pd.DataFrame:
    return pd.read_csv(path, **opzioni)


print(ordine("pizza", impasto="integrale", extra="olive"))
leggi_csv("../Dati/impianti_fv.csv", usecols=["comune", "kwp"], nrows=3)

<a id="sez-7"></a>
## 7. Esercizi

Gli esercizi usano `carico`, il carico Terna della zona Nord 2024 ordinato e senza i duplicati
dell'ora legale, come nel notebook 08, e `fig_carico`, il grafico di gennaio dell'esercizio 9.1.
La cella qui sotto li ricostruisce.

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date")
carico = carico.drop_duplicates(subset="Date")

gennaio = carico[carico["Date"].dt.month == 1]
fig_carico = px.line(gennaio, x="Date", y="Total Load [MW]", title="Carico Nord, gennaio 2024")
fig_carico.show()

<a id="es-a2-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio A2.1 · Il grafico del carico in HTML**

Il grafico del carico di gennaio, `fig_carico`, va mandato a un collega che non ha Python.

**Cosa fare.**

Salva `fig_carico` in `carico_gennaio.html` con `include_plotlyjs="cdn"`, poi aprilo con un
doppio clic dalla cartella del notebook. Output atteso: un file leggero, sotto 1 MB.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione A2.1**

Senza `include_plotlyjs="cdn"` il file contiene tutta la libreria Plotly (qualche MB) e si apre anche senza rete.

</div>

In [ ]:
fig_carico.write_html("carico_gennaio.html", include_plotlyjs="cdn")

In [ ]:
from corso import verifica
verifica("A2.1")

<a id="es-a2-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio A2.2 · La media mobile su 24 ore (facoltativo)**

Il carico quartorario oscilla molto tra giorno e notte: una media mobile su 24 ore mostra l'andamento di fondo.

**Cosa fare.**

1. Calcola in `media_24h` la media mobile su 24 ore di `Total Load [MW]`, con `Date` come indice.
2. Metti in `picco_24h` il timestamp in cui la media mobile è più alta.

*Suggerimento:* 24 ore sono 96 quarti d'ora.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione A2.2**

</div>

In [ ]:
serie = carico.set_index("Date")["Total Load [MW]"]
media_24h = serie.rolling(96).mean()
picco_24h = media_24h.idxmax()

picco_24h

In [ ]:
from corso import verifica
verifica("A2.2")

---

**Fine.**

<sub>Approfondimenti: se la classe è avanti · Giornata 2 · Aula Avanzata</sub>